# 🧠 Predicción de Accidente Cerebrovascular (ACV/Stroke)
## Taller Pre-Congreso CNIB 2025

---

> **¿Qué es un ACV (Stroke)?** Un accidente cerebrovascular ocurre cuando el flujo de sangre al cerebro se interrumpe (ACV isquémico, ~87% de casos) o cuando un vaso se rompe (ACV hemorrágico). Es la 2ª causa de muerte y la 1ª causa de discapacidad adquirida en el mundo.

> **El tiempo es cerebro 🕐:** Cada minuto sin tratamiento, el cerebro pierde ~1.9 millones de neuronas. Identificar pacientes en riesgo ANTES del evento es crítico.

---


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import numpy as np
%matplotlib inline
import seaborn as sns
plt.rcParams['figure.figsize'] = (5, 5)

## 📑 Dataset: Healthcare Dataset Stroke

**Variables del dataset:**
| Variable | Descripción médica |
|---|---|
| id | Identificador único del paciente |
| gender | Sexo del paciente |
| age | Edad en años |
| hypertension | ¿Tiene hipertensión arterial? (0/1) |
| heart_disease | ¿Tiene enfermedad cardíaca? (0/1) |
| ever_married | ¿Ha estado casado? |
| work_type | Tipo de trabajo |
| Residence_type | Rural o urbano |
| avg_glucose_level | Nivel promedio de glucosa en sangre |
| bmi | Índice de Masa Corporal |
| smoking_status | Estado de tabaquismo |
| **stroke** | **Variable objetivo: 1=tuvo ACV, 0=no tuvo ACV** |

**Factores de riesgo conocidos para ACV:**
- Hipertensión arterial (el más importante — duplica el riesgo)
- Diabetes (glucosa alta)
- Tabaquismo
- Fibrilación auricular (enfermedad cardíaca)
- Edad > 55 años
- IMC elevado (obesidad)


In [ ]:
import kagglehub
import os

# https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")
print("Ruta al dataset:", path)
print(os.listdir(path))

In [ ]:
data = pd.read_csv(os.path.join(path, "healthcare-dataset-stroke-data.csv"))
data

---

## 📊 Análisis Exploratorio (EDA)


In [ ]:
# Dimensiones del dataset: (filas, columnas)
data.shape

In [ ]:
data.columns

In [ ]:
# Tipos de datos y valores nulos
# Busca columnas con valores non-null < total → hay nulos
data.info()

In [ ]:
# Conteo de valores nulos por columna
# BMI es la única con nulos en este dataset
data.isnull().sum()

---

## 🧹 Manejo de Valores Nulos — IMC (BMI)

El BMI tiene 201 valores faltantes (~4% del dataset). Estrategia: imputar con la **media** de la columna.

> **Consideración médica:** En un estudio real, usaríamos la media *estratificada* por sexo, edad y grupo étnico, ya que el IMC "normal" varía significativamente entre grupos demográficos. Sin embargo, para este ejercicio la media global es suficiente.


In [ ]:
# Distribución del BMI antes de imputar
data['bmi'].value_counts()

In [ ]:
data['bmi'].describe()

In [ ]:
# Imputamos los nulos con la media del BMI
data['bmi'] = data['bmi'].fillna(data['bmi'].mean())
print("BMI después de imputar:")
data['bmi'].describe()

In [ ]:
# Verificamos que ya no haya nulos
data.isnull().sum()

In [ ]:
# Eliminamos la columna 'id' — no tiene valor predictivo
# (es solo un número de registro, no una característica clínica)
data.drop('id', axis=1, inplace=True)
data

---

## 📦 Detección de Outliers (Valores Atípicos)

Un **outlier** es un valor inusualmente alejado del resto. En medicina:
- Puede ser un error de medición (BMI de 0)
- Puede ser un caso genuinamente extremo (BMI de 80 en obesidad mórbida severa)

El **boxplot** muestra:
- La caja: percentiles 25-75 (rango intercuartílico, IQR)
- La línea central: mediana
- Los bigotes: valores dentro de 1.5×IQR
- Los puntos externos: outliers


In [ ]:
from matplotlib.pyplot import figure
figure(num=None, figsize=(8, 6), dpi=80, facecolor='w', edgecolor='k')

In [ ]:
# Boxplot de todas las variables numéricas
# Variables con muchos puntos por encima del bigote superior → outliers positivos
data.plot(kind='box')
plt.xticks(rotation=45, ha='right')
plt.title('Distribución y Outliers por Variable')
plt.tight_layout()
plt.show()

---

## 📊 Visualización de Variables Categóricas vs. ACV


In [ ]:
# Gráfica de barras: distribución de variables categóricas según si hubo ACV o no
# Preguntas que responde:
# - ¿Los fumadores tienen más ACV?
# - ¿Los trabajadores independientes tienen más riesgo?
# - ¿Hay diferencia entre zonas rurales y urbanas?

Patient_status = ['gender', 'work_type', 'Residence_type', 'smoking_status']
fig, axs = plt.subplots(2, 2, figsize=(10, 8))
axs = axs.flatten()

for i, feature in enumerate(Patient_status):
    sns.countplot(data=data, x=feature, hue='stroke', ax=axs[i])
    axs[i].set_title(f'{feature} vs Estado de ACV')
    axs[i].set_xlabel(feature)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## 🔤 Codificación de Variables Categóricas (Label Encoding)

Los algoritmos de ML requieren entradas numéricas. Convertimos las variables de texto a números usando `LabelEncoder`.

> **Nota técnica:** Para variables con más de 2 categorías sin orden natural (como `work_type`), idealmente deberíamos usar *One-Hot Encoding* para no introducir un orden artificial. Sin embargo, para este ejercicio, LabelEncoder es suficiente.


In [ ]:
data.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()

# Codificamos cada columna categórica de texto a número
gender         = enc.fit_transform(data['gender'])
smoking_status = enc.fit_transform(data['smoking_status'])
work_type      = enc.fit_transform(data['work_type'])
Residence_type = enc.fit_transform(data['Residence_type'])
ever_married   = enc.fit_transform(data['ever_married'])

In [ ]:
# Reemplazamos las columnas originales con las versiones codificadas
data['work_type']      = work_type
data['ever_married']   = ever_married
data['Residence_type'] = Residence_type
data['smoking_status'] = smoking_status
data['gender']         = gender

In [ ]:
# Verificamos que todas las columnas son numéricas
data

In [ ]:
data.info()

---

## 📊 Análisis de las Variables Numéricas


In [ ]:
# Distribución de edad — variable clave: el ACV aumenta exponencialmente con la edad
plt.figure(figsize=(7, 6))
sns.histplot(data['age'], bins=10, kde=True)
plt.title('Distribución de Edad en el Dataset')
plt.xlabel('Edad (años)')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
# Scatter plot: Glucosa vs BMI coloreado por ACV
# Hipótesis: pacientes con glucosa alta Y BMI alto tienen más ACV
plt.figure(figsize=(10, 6))
sns.scatterplot(data=data, x='avg_glucose_level', y='bmi', hue='stroke', alpha=0.7)
plt.title('IMC vs Glucosa y ACV')
plt.xlabel('Nivel Promedio de Glucosa')
plt.ylabel('IMC (BMI)')
plt.legend(title='ACV', labels=['No', 'Sí'])
plt.show()

---

## 🔗 Mapa de Correlaciones


In [ ]:
corrMatrix = data.corr()

plt.figure(figsize=(8, 7))
sns.heatmap(corrMatrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=2)
plt.title('Mapa de Correlación')
plt.show()

# Observa la última fila/columna (stroke):
# ¿Qué variables tienen mayor correlación con el ACV?

---

## ✂️ División de Datos y Escalado


In [ ]:
X = data.drop('stroke', axis=1)
Y = data['stroke']

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Escalado estándar: recomendado para SVM, KNN, redes neuronales
scaler_std = StandardScaler()
X_scaled   = scaler_std.fit_transform(X)

# Escalado MinMax: alternativa para redes neuronales
scaler_mm  = MinMaxScaler()
X_mm       = scaler_mm.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split

# División estratificada (importante: el ACV es raro → solo ~5% de casos)
X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y, test_size=0.2, stratify=Y, random_state=1)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.2, stratify=Y_train, random_state=1)

labels = ['Train', 'Validation', 'Test']
sizes  = [len(Y_train), len(Y_val), len(Y_test)]
plt.figure(figsize=(5,5))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90,
        colors=['#ff9999', '#66b3ff', '#99ff99'])
plt.title('División del Dataset')
plt.show()

In [ ]:
print('X_train:', X_train.shape)
print('Y_train:', Y_train.shape)
print('X_val:', X_val.shape)
print('Y_val:', Y_val.shape)
print('X_test:', X_test.shape)
print('Y_test:', Y_test.shape)
print("\nProporción de ACV en train:", Y_train.mean().round(4))

---

## 🤖 Entrenamiento de Modelos

Entrenamos 4 modelos para predecir el riesgo de ACV.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

knn_model = KNeighborsClassifier()
svm_model = SVC()
dt_model  = DecisionTreeClassifier()
rf_model  = RandomForestClassifier()

ML_models = [knn_model, svm_model, dt_model, rf_model]

In [ ]:
# Entrenamos todos los modelos
for model in ML_models:
    model.fit(X_train, Y_train)
    print(f"✓ {model.__class__.__name__} entrenado")

In [ ]:
# Métricas de accuracy para los 3 conjuntos
for model in ML_models:
    print(f"{model.__class__.__name__}")
    print(f"  Train: {model.score(X_train, Y_train):.3f} | "
          f"Val: {model.score(X_val, Y_val):.3f} | "
          f"Test: {model.score(X_test, Y_test):.3f}")
    print()

In [ ]:
from sklearn.metrics import classification_report

# Matrices de confusión y reportes de clasificación para cada modelo
# En ACV, el Recall es CRÍTICO: queremos detectar todos los que SÍ tuvieron ACV
# Un Falso Negativo (no detectar ACV) puede ser fatal

for model in ML_models:
    Y_pred = model.predict(X_test)
    confusion_matrix = metrics.confusion_matrix(Y_test, Y_pred)
    cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=confusion_matrix,
                                                 display_labels=[False, True])
    cm_display.plot(cmap='Blues', values_format='d')
    plt.title(model.__class__.__name__)
    plt.show()

    print(f"--- Reporte para {model.__class__.__name__} ---")
    print(classification_report(Y_test, Y_pred, zero_division=0))
    print("-" * 50)

In [ ]:
# Ranking de modelos por accuracy en test
ordered_models = sorted(ML_models,
    key=lambda m: m.score(X_test, Y_test), reverse=True)

print("Ranking por accuracy:")
for i, m in enumerate(ordered_models, 1):
    print(f"  {i}. {m.__class__.__name__}: {m.score(X_test, Y_test):.3f}")

In [ ]:
# Curvas ROC para comparar los modelos visualmente
# AUC (Área Bajo la Curva): 1.0 = modelo perfecto, 0.5 = aleatorio

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.svm import SVC

plt.figure(figsize=(7, 6))

for model in ML_models:
    # SVC por defecto no tiene predict_proba → reentrenar con probability=True
    if isinstance(model, SVC) and not hasattr(model, 'predict_proba'):
        model = SVC(probability=True, random_state=1)
        model.fit(X_train, Y_train)

    model_name = model.__class__.__name__

    if hasattr(model, 'predict_proba'):
        Y_prob = model.predict_proba(X_test)[:, 1]
    else:
        print(f"El modelo {model_name} no tiene predict_proba, se omite.")
        continue

    roc_auc      = roc_auc_score(Y_test, Y_prob)
    fpr, tpr, _  = roc_curve(Y_test, Y_prob)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {roc_auc:.2f})")

# Línea diagonal = clasificador aleatorio (AUC = 0.50)
plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio (AUC = 0.50)')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR / Recall)')
plt.title('Curvas ROC — Predicción de ACV')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

---

## ⚖️ Desbalanceo de Clases y SMOTE

**El problema del desbalanceo en ACV:**

En este dataset, solo ~5% de los pacientes tuvieron un ACV. Un modelo que siempre prediga "no ACV" tendrá ~95% de accuracy pero recall de 0% para la clase positiva — completamente inútil desde el punto de vista clínico.

**SMOTE resuelve esto** creando ejemplos sintéticos de ACV interpolando entre casos reales.

> **Reflexión ética 🤔:** En predicción de ACV, un Falso Positivo (decir que alguien tendrá ACV cuando no lo tendrá) causa ansiedad innecesaria y posibles tratamientos preventivos agresivos. Un Falso Negativo (no detectar el ACV que vendrá) puede costarle la vida al paciente. La elección del umbral de decisión es una decisión clínica, no solo técnica.


In [ ]:
!pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd

print("Distribución ANTES de SMOTE:")
print(pd.Series(Y_train).value_counts())

smote = SMOTE(random_state=42)
X_train_res, Y_train_res = smote.fit_resample(X_train, Y_train)

print("\nDistribución DESPUÉS de SMOTE:")
print(pd.Series(Y_train_res).value_counts())

---

## 🔧 Optimización de Hiperparámetros con GridSearchCV

Usamos **GridSearchCV** con `scoring='recall'` en lugar de accuracy. ¿Por qué?

Para ACV, queremos maximizar el **recall** (sensibilidad): detectar el mayor número posible de pacientes que SÍ tendrán un ACV, aunque aceptemos más falsos positivos.

> **Fórmula del recall:** Recall = TP / (TP + FN) — "de todos los que sí tenían ACV, ¿cuántos detecté?"


In [ ]:
from sklearn.model_selection import GridSearchCV

models_and_params = {
    "K-NN": {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [5, 9, 13],
            'weights': ['uniform', 'distance'],
        }
    },
    "SVM": {
        'model': SVC(probability=True, random_state=42),
        'params': {
            'C': [0.1, 1],
            'kernel': ['rbf', 'linear'],
        }
    },
    "Decision Tree": {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': [None, 5, 10],
            'min_samples_leaf': [1, 2]
        }
    },
    "Random Forest": {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'max_depth': [5, 10],
        }
    }
}

best_models = {}

print("Optimizando hiperparámetros con Grid Search (scoring=recall)...")
for name, config in models_and_params.items():
    grid_search = GridSearchCV(
        config['model'], config['params'],
        cv=5, scoring='recall', n_jobs=-1
    )
    grid_search.fit(X_train_res, Y_train_res)
    best_models[name] = grid_search.best_estimator_
    print(f"\n{name}")
    print(f"  Mejor Recall (CV): {grid_search.best_score_:.4f}")
    print(f"  Mejores Parámetros: {grid_search.best_params_}")

# Evaluación final con las curvas ROC
plt.figure(figsize=(10, 8))
print("\n--- Evaluación Final de Modelos Optimizados ---")

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n--- {name} ---")
    print(classification_report(Y_test, y_pred, zero_division=0))

    roc_auc     = roc_auc_score(Y_test, y_prob)
    fpr, tpr, _ = roc_curve(Y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio (AUC = 0.50)')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR / Recall)')
plt.title("Curvas ROC — Modelos Optimizados para Predicción de ACV")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()